# End-to-End Traditional RAG Pipeline: PDF Ingestion → Vector DB → LLM

This notebook implements the complete **Retrieval-Augmented Generation (RAG)** pipeline for the `TraditionalRAG` codebase, taking raw PDF documents on disk all the way to context-grounded LLM answers.

## Pipeline Architecture
```text
PDF Files (../data/**/*.pdf)
   │
   ▼  1. Ingestion (PyPDFLoader)
Page-Level LangChain Documents
   │
   ▼  2. Chunking (RecursiveCharacterTextSplitter: size=1000, overlap=200)
Overlapping Text Chunks
   │
   ▼  3. Embedding (SentenceTransformer: all-MiniLM-L6-v2, 384-dim)
Dense Vector Embeddings
   │
   ▼  4. Indexing (ChromaDB PersistentClient at ../vector_stores/chroma)
Persistent Vector Collection ("pdf_documents")
   │
   ▼  5. Basic Generation (rag_simple)
Direct Context-Grounded Answer
   │
   ▼  6. Enhanced RAG (rag_advanced)
Answers + Confidence Scores + Source Attribution + Context Returns
   │
   ▼  7. Production Advanced Pipeline (AdvancedRAGPipeline)
Streaming UI + Formal Citations + Conversation History + Auto-Summaries
```

## Prerequisites
- PDF documents located under `../data/` (e.g., `../data/pdf/`).
- A `.env` file in the project root containing `GROQ_API_KEY` for the LLM generation stage.
- Dependencies installed from `pyproject.toml` / `requirements.txt` (`langchain-community`, `langchain-text-splitters`, `pypdf`, `sentence-transformers`, `chromadb`, `langchain-groq`, `python-dotenv`).

## 1. Data Ingestion: Loading PDF Documents

In this stage, we import the document-loading and text-splitting utilities and define `process_all_pdfs(pdf_directory)` to:
1. Recursively discover all `.pdf` files under a directory using `Path.glob("**/*.pdf")`.
2. Parse each PDF into per-page LangChain `Document` objects via `PyPDFLoader`.
3. Enrich each page's metadata with `source_file` (filename) and `file_type` (`"pdf"`).
4. Isolate per-file errors so a single corrupt or unreadable PDF does not halt the entire ingestion run.

In [1]:
# Standard library utilities for filesystem operations and path traversal
import os
# LangChain loader for extracting page-by-page text from PDF files
from langchain_community.document_loaders import PyPDFLoader
# Splitter for breaking long documents into overlapping chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/y2/0wvpjnds5dzdwvy9j5j1h0zh0000gn/T/ipykernel_13621/4145479051.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
### Read all the PDFs inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    # Accumulator for all page-level Document objects across all discovered PDFs
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            # Load the PDF file; PyPDFLoader returns one Document per page
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            # Append all pages from this PDF to the master document list
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            # Log failure for this file and continue processing remaining PDFs
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: TCS NQT Coding Questions Hub.pdf
  ✓ Loaded 3 pages

Processing: Debkanta Study Zone - TCS NQT 2027 Complete Placement Guide (1).pdf
  ✓ Loaded 12 pages

Total documents loaded: 15


### Inspect Loaded Page Documents
Display `all_pdf_documents` to verify the extracted `page_content` and enriched `metadata` (`source`, `page`, `page_label`, `total_pages`, `source_file`, `file_type`) for every loaded PDF page.

In [3]:
# Inspect the full list of loaded page-level Document objects
all_pdf_documents

[Document(metadata={'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-11-10T13:09:40+00:00', 'title': 'TCS NQT Coding Questions - Debkanta Study Zone', 'moddate': '2026-11-10T13:09:40+00:00', 'source': '../data/pdf/TCS NQT Coding Questions Hub.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'TCS NQT Coding Questions Hub.pdf', 'file_type': 'pdf'}, page_content='DEBKANTA STUDY ZONE\nTCS PLACEMENT PREPARATION HUB — 2025/2026 EDITION\nTCS NQT Advanced Coding Questions Practice Sheet (Questions Only)\nSlot 1 & Slot 2 — Core Array & Logic Problems\n1. Move Empty Packets (Zeros) to End September Day 1 • Slot 1\nA chocolate factory packs chocolates into packets represented as an array of N integers. Find the\nempty packets (represented as 0) and push them all to the end of the conveyor belt (array) while\npreserving the relative order of non-zero elements.\nEXAMPLE TEST CASE\nInput: N = 8, Array = [4, 5, 0, 1, 9, 0, 5, 0]\nOutput: 4 5 1 9 5 0 0 0\nP

## 2. Text Splitting & Chunking

Full PDF pages are often too long or contain multiple unrelated topics for precise semantic search. `split_documents()` uses `RecursiveCharacterTextSplitter` to break pages into smaller overlapping chunks:
- **`chunk_size=1000`**: Maximum target size of each chunk (measured in characters via `length_function=len`).
- **`chunk_overlap=200`**: Number of overlapping characters carried over between consecutive chunks so facts spanning a split boundary are not lost.
- **`separators=["\n\n", "\n", " ", ""]`**: Tries to split along natural boundaries first (paragraphs → lines → words → individual characters) while preserving original page metadata on every child chunk.

In [4]:
### Text Splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """ Split documents into smaller chunks for better RAG performance. """
    # Configure recursive splitting to prefer paragraph/line/word boundaries before hard character cuts
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    # Split input Document objects while automatically propagating metadata to each chunk
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show a sample of a chunk
    if split_docs:
        print(f"\nExample Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs



In [5]:
# Split the loaded PDF pages into chunks using the default parameters (1000 chars, 200 overlap)
chunks = split_documents(all_pdf_documents)
# Display the resulting chunked Document list
chunks

Split 15 documents into 42 chunks

Example Chunk:
Content: DEBKANTA STUDY ZONE
TCS PLACEMENT PREPARATION HUB — 2025/2026 EDITION
TCS NQT Advanced Coding Questions Practice Sheet (Questions Only)
Slot 1 & Slot 2 — Core Array & Logic Problems
1. Move Empty Pack...
Metadata: {'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-11-10T13:09:40+00:00', 'title': 'TCS NQT Coding Questions - Debkanta Study Zone', 'moddate': '2026-11-10T13:09:40+00:00', 'source': '../data/pdf/TCS NQT Coding Questions Hub.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'TCS NQT Coding Questions Hub.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-11-10T13:09:40+00:00', 'title': 'TCS NQT Coding Questions - Debkanta Study Zone', 'moddate': '2026-11-10T13:09:40+00:00', 'source': '../data/pdf/TCS NQT Coding Questions Hub.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'TCS NQT Coding Questions Hub.pdf', 'file_type': 'pdf'}, page_content='DEBKANTA STUDY ZONE\nTCS PLACEMENT PREPARATION HUB — 2025/2026 EDITION\nTCS NQT Advanced Coding Questions Practice Sheet (Questions Only)\nSlot 1 & Slot 2 — Core Array & Logic Problems\n1. Move Empty Packets (Zeros) to End September Day 1 • Slot 1\nA chocolate factory packs chocolates into packets represented as an array of N integers. Find the\nempty packets (represented as 0) and push them all to the end of the conveyor belt (array) while\npreserving the relative order of non-zero elements.\nEXAMPLE TEST CASE\nInput: N = 8, Array = [4, 5, 0, 1, 9, 0, 5, 0]\nOutput: 4 5 1 9 5 0 0 0\nP

## 3. Embeddings & Vector Store Setup

With text chunks ready, the next stage transforms each chunk into a dense numeric vector (embedding) and stores it in a persistent vector database for fast similarity search.

### Dependencies Imported Below
- **`numpy`**: Handles dense embedding matrices returned by the embedding model.
- **`SentenceTransformer`**: Loads local HuggingFace sentence-embedding models.
- **`chromadb` & `Settings`**: Local persistent vector database for storing and querying embeddings, raw text, and metadata.
- **`uuid`**: Generates unique document IDs for ChromaDB records.
- **`typing` (`List`, `Dict`, `Any`, `Tuple`)**: Type annotations for class methods.
- **`cosine_similarity`**: Utility from `scikit-learn` for pairwise vector similarity calculations.

In [6]:
# Numerical array handling for embedding vectors
import numpy as np
# Local dense text embedding model loader
from sentence_transformers import SentenceTransformer
# Persistent vector database client and configuration
import chromadb
from chromadb.config import Settings
# Unique identifier generation for stored chunks
import uuid
# Type hints for pipeline classes and methods
from typing import List, Dict, Any, Tuple
# Pairwise cosine similarity utility
from sklearn.metrics.pairwise import cosine_similarity

/Users/raunakmanna/Documents/Programming/RAG/TraditionalRAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1120 16:03:05.644000 13621 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


### 3.1 Embedding Manager (`all-MiniLM-L6-v2`)

`EmbeddingManager` encapsulates loading and invoking a HuggingFace `SentenceTransformer` model:
- Defaults to **`all-MiniLM-L6-v2`**, a lightweight, fast embedding model producing **384-dimensional** vectors well-suited for semantic retrieval.
- `_load_model()` downloads/loads the model at initialization and reports its embedding dimension.
- `generate_embeddings(texts)` encodes a list of strings in batches and returns a 2D `np.ndarray` of shape `(len(texts), embedding_dim)`.

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        # Eagerly load the embedding model upon instantiation
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            # Downloads the model on first run (if not cached) and loads it into memory
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        # Batch-encode all input strings into dense vector representations
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

# Create a default EmbeddingManager instance using all-MiniLM-L6-v2
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6736.52it/s]


Model loaded successfully. Embedding dimension: 384


### 3.2 Persistent Vector Store (`ChromaDB`)

`VectorStore` manages a local, disk-backed ChromaDB collection:
- **`persist_directory="../vector_stores/chroma"`**: Persists the SQLite + HNSW vector index on disk so embeddings survive kernel restarts.
- **`collection_name="pdf_documents"`**: Creates or opens the target collection via `get_or_create_collection()`.
- **`add_documents(documents, embeddings)`**:
  1. Validates a 1-to-1 match between chunk `Document` objects and their computed embeddings.
  2. Generates a unique ID (`doc_<8-char-uuid>_<index>`) for each chunk.
  3. Copies each chunk's metadata and appends `doc_index` and `content_length` without mutating the original `Document`.
  4. Converts NumPy vectors to Python lists and persists IDs, embeddings, metadata, and raw text into ChromaDB.

In [8]:
class VectorStore:
    """ Manages document embeddings in a ChromDB vector store """

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../vector_stores/chroma"):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        # Set up persistent storage and open/create the ChromaDB collection
        self._initialize_store()

    def _initialize_store(self):
        """ Initialize the ChromaDB client and collection """
        try:
            # Create persistant chromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF document embeddings"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing Vector Store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        # Guard against mismatched document and embedding counts
        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            # Shallow-copy existing metadata to avoid mutating the source Document object
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            # Convert NumPy vector to standard Python list as required by ChromaDB
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(
                f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

# Initialize the persistent VectorStore using default collection name and storage directory
vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 84


### 3.3 Inspect Chunks & Populate the Vector Store

Before embedding, inspect `chunks` to confirm the split output, then run the three-step indexing flow:
1. Extract plain text (`page_content`) from every chunk.
2. Compute dense embeddings with `embedding_manager.generate_embeddings(texts)`.
3. Insert chunks and their embeddings into `vectorstore`.

In [9]:
# Quick inspection of the chunked Document objects before embedding
chunks

[Document(metadata={'producer': 'Skia/PDF m136', 'creator': 'Chromium', 'creationdate': '2026-11-10T13:09:40+00:00', 'title': 'TCS NQT Coding Questions - Debkanta Study Zone', 'moddate': '2026-11-10T13:09:40+00:00', 'source': '../data/pdf/TCS NQT Coding Questions Hub.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1', 'source_file': 'TCS NQT Coding Questions Hub.pdf', 'file_type': 'pdf'}, page_content='DEBKANTA STUDY ZONE\nTCS PLACEMENT PREPARATION HUB — 2025/2026 EDITION\nTCS NQT Advanced Coding Questions Practice Sheet (Questions Only)\nSlot 1 & Slot 2 — Core Array & Logic Problems\n1. Move Empty Packets (Zeros) to End September Day 1 • Slot 1\nA chocolate factory packs chocolates into packets represented as an array of N integers. Find the\nempty packets (represented as 0) and push them all to the end of the conveyor belt (array) while\npreserving the relative order of non-zero elements.\nEXAMPLE TEST CASE\nInput: N = 8, Array = [4, 5, 0, 1, 9, 0, 5, 0]\nOutput: 4 5 1 9 5 0 0 0\nP

In [10]:
### Convert the text chunks into embeddings
# Extract raw text strings from each LangChain Document chunk
texts = [doc.page_content for doc in chunks]

### Generate embeddings for the text chunks
# Encode all chunk texts into a 2D NumPy array of dense vectors
embeddings = embedding_manager.generate_embeddings(texts)

### Store in the Vector database
# Persist chunks, metadata, and embeddings in the ChromaDB collection
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 42 texts...


Batches: 100%|██████████| 2/2 [00:01<00:00,  1.33it/s]


Generated embeddings with shape: (42, 384)
Adding 42 documents to vector store...
Successfully added 42 documents to vector store
Total documents in collection: 126


## 4. Semantic Retrieval Pipeline (`RAGRetriever`)

`RAGRetriever` connects the `EmbeddingManager` and `VectorStore` to perform semantic nearest-neighbor search for natural-language queries:
1. Embeds the user query using the **same** `SentenceTransformer` model used at indexing time so query and document vectors live in the same embedding space.
2. Queries ChromaDB (`collection.query`) for the `top_k` closest chunks.
3. Converts each returned vector `distance` into a similarity score (`similarity_score = 1 - distance`) and filters out results below `score_threshold`.
4. Returns a ranked list of dictionaries containing `id`, `content`, `metadata`, `similarity_score`, `distance`, and `rank`.

> **Note:** ChromaDB's `query()` returns batched nested lists (one sublist per input query), so index `[0]` unpacks the results for the single query string.

In [11]:
class RAGRetriever:
    """ Handles Query based retrieval from the vector store """

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the RAG retriever

        Args:
            vector_store: Instance of VectorStore
            embedding_manager: Instance of EmbeddingManager
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        # Extract the single vector [0] from the batch embedding output
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            # Query ChromaDB for the top_k nearest chunks
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            # Unpack the first (and only) query batch from ChromaDB's nested result structure
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    # Keep only results that satisfy the minimum similarity threshold
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(
                    f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

# Wire the persistent VectorStore and EmbeddingManager into the retriever
rag_retriever = RAGRetriever(vectorstore,embedding_manager)

### 4.1 Testing Semantic Retrieval

Run sample queries against `rag_retriever` to verify that relevant chunks from the indexed PDF documents are returned along with their similarity scores and metadata.

In [12]:
# Sample retrieval query 1: look up the exam sections overview
rag_retriever.retrieve("What is the Overview of the 5 Exam Sections?")

Retrieving documents for query: 'What is the Overview of the 5 Exam Sections?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.10it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_27171500_7',
  'content': 'tracker).\nOverview of the 5 Exam Sections\n# Section Name Part Topics Covered Questions / Tasks Time Allotted\n01 Numerical Ability Part A:\nFoundation\n15 Topics 20 Questions 25 Minutes (~75\nsec/Q)\n02 Reasoning Ability Part A:\nFoundation\n13 Topics 20 Questions 25 Minutes (~75\nsec/Q)\n03 Verbal Ability (2027\nUpdated)\nPart A:\nFoundation\n5 Task Areas 25+ Tasks 25 Minutes (Timed\nper item)\n04 Advanced Quantitative +\nReasoning\nPart B:\nAdvanced\n10 Topics 14 Questions 25 Minutes (~107\nsec/Q)\n05 Advanced Coding Part B:\nAdvanced\n16 Topics 2 Problems (Easy\n+ Med)\n90 Minutes (35 +\n55 min)\nDebkanta Study Zone — TCS NQT 2027 Complete Placement Guide Page 1 of 12',
  'metadata': {'content_length': 652,
   'total_pages': 12,
   'doc_index': 7,
   'file_type': 'pdf',
   'title': 'Debkanta Study Zone — TCS NQT 2027 Complete Placement Guide',
   'moddate': '2026-11-10T03:23:09+00:00',
   'creator': 'Chromium',
   'page_label': '1',
   'sour

In [13]:
# Sample retrieval query 2: look up roles, CTC packages, and eligibility criteria
rag_retriever.retrieve("what is Roles, Packages (CTC) & Eligibility Criteria?")

Retrieving documents for query: 'what is Roles, Packages (CTC) & Eligibility Criteria?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.09it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_0e9c1fbd_10',
  'content': 'Offer\nTier\nQualification\nCriteria\nExperience 0–\n1 Year (UG)\nExperience 0–\n1 Year (PG)\nExperience 1–2\nYears (UG)\nExperience 1–2\nYears (PG)\nTCS\nPrime\nTop Foundation +\nAdvanced Score\n9.09 – 9.30 LPA 11.59 – 11.80\nLPA\n9.45 – 9.66 LPA 12.05 – 12.26\nLPA\nTCS\nDigital\nStrong Foundation\n+ Advanced Score\n7.09 – 7.30 LPA 7.39 – 7.60 LPA 7.50 – 7.72 LPA 7.82 – 8.04 LPA\nTCS\nNinja\nEntry Role via\nFoundation Score\nEntry-level CTC (Exact figure confirmed on the official TCS NextStep\nnotification for your cycle)\nEligibility Criteria:\n• Courses: B.Tech / B.E, M.Tech / M.E, MCA, M.Sc / MS in any specialization from AICTE/UGC recognized\nuniversities or colleges.\n• Batch: 2027 Batch is 100% eligible (off-campus hiring). 2026 and 2025 batches depend on TCS criteria\nfor that specific hiring cycle.\n1.4 8-Week Fast-Track Preparation Plan\nWeeks Primary Focus\nWeeks 1–2 Numerical & Reasoning basics, topic-wise timed drills.\nWeek 3 Verba

## 5. LLM Integration: Context-Augmented Generation with Groq

In the final **Generation** stage, retrieved context chunks from `RAGRetriever` are injected into a prompt and sent to a Groq-hosted LLM (`ChatGroq`) to synthesize a grounded natural-language answer.

> **Environment Requirement:** Ensure `GROQ_API_KEY` is defined in the project `.env` file before executing the cells below.

### 5.1 List Available Groq Models

Query Groq's OpenAI-compatible `/openai/v1/models` endpoint to confirm `GROQ_API_KEY` authentication and inspect the model IDs currently accessible to your account.

In [14]:
import os
import requests
from dotenv import load_dotenv

# Load environment variables (including GROQ_API_KEY) from the .env file
load_dotenv()
api_key = os.getenv("GROQ_API_KEY")

# Authenticate via Bearer token and fetch the list of available Groq models
headers = {"Authorization": f"Bearer {api_key}"}
response = requests.get(
    "https://api.groq.com/openai/v1/models", headers=headers)

# Print each available model identifier returned by the API
for model in response.json().get("data", []):
    print(model["id"])

whisper-large-v3-turbo
canopylabs/orpheus-v1-english
openai/gpt-oss-120b
openai/gpt-oss-20b
openai/gpt-oss-safeguard-20b
qwen/qwen3.8-27b
allam-2-7b
whisper-large-v3
canopylabs/orpheus-arabic-saudi
meta-llama/llama-prompt-guard-2-22m
meta-llama/llama-prompt-guard-2-86m


### 5.2 Initialize `ChatGroq` & Define `rag_simple`

Configure the Groq chat model and define the end-to-end `rag_simple()` helper:
- **`model="qwen/qwen3.8-27b"`**: Target Groq-hosted LLM.
- **`temperature=0.1`**: Low temperature to keep answers factual and closely grounded in the retrieved context.
- **`max_tokens=1024`**: Upper bound on generated response length.
- **`rag_simple(query, retrieverm, llm, top_k=3)`**:
  1. Retrieves the `top_k` most relevant chunks from the vector store.
  2. Concatenates their text into a single `context` block (returning a fallback message if nothing matches).
  3. Builds a grounded prompt pairing `context` with `query`, invokes the LLM, and returns `response.content`.

In [15]:
### Simple RAG Pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

### Initialize the Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")

# Instantiate the Groq-backed chat model with low temperature for grounded QA
llm = ChatGroq(api_key=groq_api_key, model="qwen/qwen3.8-27b",
               temperature=0.1, max_tokens=1024)

## Simple RAG Pipeline
def rag_simple(query, retrieverm, llm, top_k = 3):
    ## retrieve the context
    # Fetch the top_k most relevant chunks from ChromaDB
    results = retrieverm.retrieve(query, top_k=top_k)
    # Concatenate retrieved chunk contents into a single context string
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    # Gracefully exit if no relevant context was retrieved
    if not context:
        return "No relevant context found for the query."

    ## Generate the answer using the LLM
    # Construct a grounded prompt combining the retrieved context and user query
    prompt = f"""Based on this context: {context}
        Question: {query}
        Answer:"""

    # Invoke the LLM and return the generated text content
    response = llm.invoke([prompt.format(context = context, query=query)])
    return response.content

### 5.3 Run an End-to-End RAG Query

Execute `rag_simple()` with a sample question to retrieve supporting PDF chunks and print the LLM's final context-grounded answer.

In [21]:
# Run an end-to-end RAG query combining ChromaDB retrieval with Groq LLM generation
answer = rag_simple(
    "what is Roles, Packages (CTC) & Eligibility Criteria?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'what is Roles, Packages (CTC) & Eligibility Criteria?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.59it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Based on the context provided, here are the Roles, Packages (CTC), and Eligibility Criteria for TCS NextStep:

### **1. Roles & Packages (CTC)**

The offer tiers are determined by the candidate's score in the Foundation and Advanced assessments.

| Offer Tier | Qualification Criteria | Experience 0–1 Year (UG) | Experience 0–1 Year (PG) | Experience 1–2 Years (UG) | Experience 1–2 Years (PG) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **TCS Prime** | Top Foundation + Advanced Score | 9.09 – 9.30 LPA | 11.59 – 11.80 LPA | 9.45 – 9.66 LPA | 12.05 – 12.26 LPA |
| **TCS Digital** | Strong Foundation + Advanced Score | 7.09 – 7.30 LPA | 7.39 – 7.60 LPA | 7.50 – 7.72 LPA | 7.82 – 8.04 LPA |
| **TCS Ninja** | Entry Role via Foundation Score | Entry-level CTC* | Entry-level CTC* | Entry-level CTC* | Entry-level CTC* |

*\*Note: The exact figure for TCS Ninja is confirmed on the official TCS NextStep notificat

## 6. Enhanced RAG Pipeline (`rag_advanced`)

While `rag_simple()` demonstrates the basic retrieve-and-generate flow, production applications require metadata visibility, provenance tracking, and confidence scoring. 

The `rag_advanced()` function extends the pipeline with several capabilities:
- **Structured Response Payload**: Returns a structured dictionary containing `answer`, `sources`, `confidence`, and optionally full `context`.
- **Source Attribution & Citations**: Extracts the `source_file`, `page` number, similarity `score`, and a text preview for each chunk used in generation.
- **Confidence Scoring**: Computes the maximum similarity score among retrieved chunks as an indicator of retrieval confidence.
- **Score Thresholding (`min_score`)**: Filters out low-relevance chunks to prevent context pollution and hallucinations.

> [!NOTE]
> **Understanding Distance Metrics & `min_score`**:
> ChromaDB defaults to squared $L_2$ (Euclidean) distance (`"space": "l2"`), and `RAGRetriever` calculates similarity as `1 - distance`. Because `SentenceTransformer` vectors are unit-normalized, the squared Euclidean distance is related to cosine similarity by $d_{L2}^2 = 2(1 - \cos(	heta))$. For typical relevant matches, $d_{L2}^2 pprox 0.95$–$1.0$, producing a similarity score of around $0.0$ to $0.05$. Therefore, keep `min_score=0.0` (or low values like `0.05`) so valid documents are not inadvertently filtered out.

In [23]:
# --- Enhanced RAG Pipeline Features ---
# Advanced function-based RAG pipeline returning structured metadata, confidence scores, and source attribution
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    # Step 1: Retrieve context chunks filtered by the minimum similarity score threshold
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    # Early exit fallback if no candidate chunks satisfy the score threshold
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources
    # Join chunk contents into a single concatenated context string for the prompt
    context = "\n\n".join([doc['content'] for doc in results])
    # Extract provenance metadata (source file, page, score, and text snippet) for each chunk
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    # Assign the top similarity score as the overall retrieval confidence
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    # Format the prompt with retrieved context and user query
    prompt = f"""Use the following context to answer the question concisely.
        Context:
        {context}
        Question: {query}
        Answer:"""
    # Invoke the LLM with the context-grounded prompt
    response = llm.invoke([prompt])

    # Construct the structured response payload
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    # Optionally append the raw concatenated context string if requested by the caller
    if return_context:
        output['context'] = context
    return output


# Example usage:
# Query the enhanced RAG pipeline with min_score=0.0 to accommodate ChromaDB L2 distance scaling
result = rag_advanced("How to start preparing for NQT?", rag_retriever,
                      llm, top_k=3, min_score=0.0, return_context=True)
# Display generated answer, citations, confidence score, and context preview
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'How to start preparing for NQT?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.07it/s]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: To start preparing for the TCS NQT 2027, you should follow the **90-Day First-Attempt Roadmap** provided in the guide. This involves:

1.  **Adopting the Daily Routine:** Follow the structured daily schedule outlined in the 13-week plan.
2.  **Using Topic Checklists:** Work through the 59+ topics across the 5 exam sections (Numerical Ability, Reasoning, English, Coding, etc.) according to their priority levels.
3.  **Applying Section Strategies:** Use specific strategies for each section to optimize your performance.
4.  **Tracking Progress:** Utilize the 90-day tracker to monitor your preparation consistently.
Sources: [{'source': 'Debkanta Study Zone - TCS NQT 2027 Complete Placement Guide (1).pdf', 'page': 0, 'score': 0.3025133013725281, 'preview': '2027 BATCH • OFF-CAMPUS & NQT PLACEMENT EDITION\nTCS NQT 2027 Complete Placement\nGuide\nExam Pattern, Roles & Packages, Eligibility Rules, Complet

## 7. Advanced RAG Pipeline with Streaming, Citations, History & Summarization (`AdvancedRAGPipeline`)

`AdvancedRAGPipeline` packages the retrieval and generation components into an extensible, production-oriented class that integrates four enterprise-grade capabilities:

1. **Stateful Conversation History (`self.history`)**: Tracks all multi-turn interactions, maintaining an audit trail of past queries, answers, attributed sources, and generated summaries.
2. **Bibliographic In-Text Citations**: Automatically constructs numbered citation references (`[1] filename.pdf (page X)`) appended directly to the final answer.
3. **Simulated Token Streaming (`stream=True`)**: Emulates progressive, chunked response streaming for responsive user interface rendering.
4. **Automated Secondary Summarization (`summarize=True`)**: Employs a multi-stage LLM chaining pattern—the primary generation produces a comprehensive grounded answer, and a secondary prompt invokes the LLM to distill a 2-sentence executive summary.

In [25]:
import time

# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---

class AdvancedRAGPipeline:
    # Production-ready class orchestrating retrieval, simulated streaming, citation formatting, and chat history
    def __init__(self, retriever, llm):
        # Store retriever and LLM instances
        self.retriever = retriever
        self.llm = llm
        # Stateful conversation history log storing past questions, answers, sources, and summaries
        self.history = []

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        stream: bool = False,
        summarize: bool = False
    ) -> Dict[str, Any]:
        # Retrieve candidate chunks from the vector store using top_k and minimum score threshold
        results = self.retriever.retrieve(
            question, top_k=top_k, score_threshold=min_score
        )

        # Fallback handling when retrieval yields no qualifying chunks
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            # Concatenate retrieved chunk contents into context string
            context = "\n\n".join(doc["content"] for doc in results)
            # Build detailed source attribution dictionaries
            sources = [
                {
                    "source": doc["metadata"].get(
                        "source_file", doc["metadata"].get("source", "unknown")
                    ),
                    "page": doc["metadata"].get("page", "unknown"),
                    "score": doc["similarity_score"],
                    "preview": doc["content"][:120] + "..."
                }
                for doc in results
            ]

            # Assemble prompt template with context and query
            prompt = f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {question}

Answer:"""

            # Optional simulated streaming to mimic token-by-token terminal rendering
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i + 80], end="", flush=True)
                    time.sleep(0.05)
                print()

            # Execute generation via ChatGroq
            response = self.llm.invoke([prompt])
            answer = response.content

        # Generate numbered bibliographic citations linking to source file and page number
        citations = [
            f"[{i + 1}] {src['source']} (page {src['page']})"
            for i, src in enumerate(sources)
        ]
        # Append formatted citations to the final answer text
        answer_with_citations = (
            answer + "\n\nCitations:\n" + "\n".join(citations)
            if citations else answer
        )

        # Optional secondary LLM call to create a concise 2-sentence executive summary
        summary = None
        if summarize and answer:
            summary_prompt = (
                f"Summarize the following answer in 2 sentences:\n{answer}"
            )
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Append turn details to internal history list for stateful tracking
        self.history.append({
            "question": question,
            "answer": answer,
            "sources": sources,
            "summary": summary
        })

        # Return full payload including citations, summary, and accumulated history
        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history
        }


# Example usage
# Instantiate the advanced pipeline with the ChromaDB retriever and ChatGroq LLM
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
# Execute a comprehensive query with streaming output and summary enabled
result = adv_rag.query(
    "How to start preparing for NQT?",
    top_k=3,
    min_score=0.1,
    stream=True,
    summarize=True
)
# Output final results: answer with citations, executive summary, and latest history record
print("\nFinal Answer:", result["answer"])
print("Summary:", result["summary"])
print("History:", result["history"][-1])

Retrieving documents for query: 'How to start preparing for NQT?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.94it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
2027 BATCH • OFF-CAMPUS & NQT PLACEMENT EDITION
TCS NQT 2027 Complete Placement
Guide
Exam Pattern, Roles & Packages, Eligibility Rules, Complete Topic-Wise Syllabus
(Foundation & Advanced), and the 90-Day First-Attempt Roadmap — all in one
structured

 handbook.
By Debkanta Study Zone — TCS Placement Guide
190 Min
TOTAL EXAM TIME
2 Parts
PART A & PART B
0
NEGATIVE MARKING
2 Problems
ADVANCED CODING
How to Use This Guide: This placement guide integrates three core resources into a single master
playbook: (1) The TCS NQT Complete Guide (exam pattern, roles, CTC, eligibility), (2) The 2027 Batch
Detailed Topic-Wise Syllabus (59+ topics across 5 sections with priority levels), and (3) The 90-Day
Roadmap to TCS NQT (daily routine, 13-week plan, topic checklists, section strategies, and 90-day
tracker).
Overview of the 5 Exam Sections
# Section Name Part Topics Covered Questions / Tasks Time Allotted
01 Numerical Ability Part A:
Foundation
15 Topics 20 Questions 25 Minutes (~75
sec/Q)

2027 BATCH • OFF-CAMPUS & NQT PLACEMENT EDITION
TCS NQT 2027 Complete Placement
Guide
Exam Pattern, Roles & Packages, Eligibility Rules, Complete Topic-Wise Syllabus
(Foundation & Advanced), and the 90-Day First-Attempt Roadmap — all in one
structured handb